# ESA SOC Regression Model (Model 1)

This notebook implements the training pipeline for **Module 1**: Predicting Soil Organic Carbon (SOC) from ESA satellite data. It mirrors the logic built into `src/model_training/m1_soc_regression.py`.

The goal is to load pre-processed ESA data, train an XGBoost Regressor, evaluate it, and save the model artifact to disk.

## 1. Import Dependencies

We use `xgboost` as our main predictive algorithm. Hyperparameters and constants are imported from our centralized `src/config`.

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import joblib

# Scikit-Learn & XGBoost
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Allow imports from src/
sys.path.append(str(Path.cwd().parent.parent))

# Import constants and hyperparameters
from src.config import (
    PROCESSED_FILES,
    WEIGHTS_DIR,
    ESA_TARGET_COL,
    MODULE1_PARAMS,
    RANDOM_SEED
)

## 2. Load Processed Data
Load the dimensionality-reduced (PCA) processed dataset generated by notebook `07_data_processing`.

In [2]:
data_path = PROCESSED_FILES["esa_train"]
print(f"Loading data from: {data_path}")

try:
    df = pd.read_csv(data_path)
    print(f"Loaded successfully! Shape: {df.shape}")
except FileNotFoundError:
    print(f"Could not find {data_path}. Ensure you have run notebook 07_data_processing.ipynb first.")
    df = None

if df is not None:
    display(df.head(3))

Loading data from: C:\Users\justi\Downloads\ML project data\Terraflux\data\processed\esa_train_processed.csv
Loaded successfully! Shape: (23898, 61)


,PC_1,PC_2,PC_3,PC_4,PC_5,PC_6,PC_7,PC_8,PC_9,PC_10,...,PC_52,PC_53,PC_54,PC_55,PC_56,PC_57,PC_58,PC_59,PC_60,logoc_d.f
0,3.772226,-3.768648,0.837889,-5.156308,-0.813787,-3.234172,-1.120613,3.444649,-1.191004,-1.053877,...,0.655750,0.049096,0.793758,-0.186165,0.425859,-0.153938,-0.647021,0.052653,-0.603093,5.220388
1,4.679112,-3.600828,-1.423638,-5.529601,-0.125095,-3.054660,-0.038824,3.845371,-2.020001,-0.388338,...,0.449869,-0.097359,0.473236,0.013520,-1.246294,-1.405060,0.696477,-0.250902,-0.842326,3.335679
2,3.898829,-3.827087,1.565726,-4.344070,-1.295705,-3.474332,-0.420283,3.799724,-1.499812,-1.592073,...,-1.202139,-0.020875,-0.216385,1.497578,-0.772020,0.206380,0.195774,-0.148379,-1.011593,3.887080


## 3. Split Train & Validation Defaults

Next, isolate our target variable (Soil Organic Carbon) from the features and split them into Training (80%) and Validation (20%) sets.

In [3]:
if df is not None:
    # 1. Split Features (X) & Target (y)
    X = df.drop(columns=[ESA_TARGET_COL])
    y = df[ESA_TARGET_COL]

    print(f"Feature Matrix: {X.shape[0]} rows, {X.shape[1]} principal components")

    # 2. Train-Validation Split (80/20)
    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.2, random_state=RANDOM_SEED
    )
    print(f"Training split: {len(X_train)} samples")
    print(f"Validation split: {len(X_val)} samples")

Feature Matrix: 23898 rows, 60 principal components
Training split: 19118 samples
Validation split: 4780 samples


## 4. Define and Train the Model
We instantiate our XGBoost regressor, injecting hyperparameter configurations from `MODULE1_PARAMS`. We also provide the validation set directly to `fit()` to display progressive learning curves (if `verbose` is enabled).

In [4]:
if df is not None:
    xgb_params = MODULE1_PARAMS.get("xgb", {})
    print("🚀 Initializing XGBoost Regressor with hyperparams:")
    for k, v in xgb_params.items():
        print(f"  - {k}: {v}")

    # Initialize the model with dynamic hyperparameter configuration and utilize all CPU cores
    kwargs = {**xgb_params, "n_jobs": -1}
    model = XGBRegressor(**kwargs)

    print("\nTraining model...")
    # Add validation set so XGBoost tracks the holdout error across trees
    model.fit(
        X_train, y_train,
        eval_set=[(X_train, y_train), (X_val, y_val)],
        verbose=100
    )
    print("Training Complete ✅")

🚀 Initializing XGBoost Regressor with hyperparams:
  - n_estimators: 300
  - max_depth: 6
  - learning_rate: 0.1
  - subsample: 0.8
  - colsample_bytree: 0.8
  - random_state: 42

Training model...
[0]	validation_0-rmse:1.00303	validation_1-rmse:1.01304
[100]	validation_0-rmse:0.49198	validation_1-rmse:0.63826
[200]	validation_0-rmse:0.38667	validation_1-rmse:0.59922
[299]	validation_0-rmse:0.31935	validation_1-rmse:0.58274
Training Complete ✅


## 5. Evaluate Performance

Finally, calculate and present error metrics for the holdout set (`X_val`). We will display standard metrics (RMSE, MAE, R-squared) as well as an intuitive percentage metric: "Accuracy within 0.5".

In [5]:
if df is not None:
    print("\n" + "="*60)
    print("🎯 Validation Performance")
    print("=" * 60)

    # Make predictions on unknown data
    y_pred = model.predict(X_val)

    # 1. Error Metrics
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    mae = mean_absolute_error(y_val, y_pred)
    r2 = r2_score(y_val, y_pred)

    # 2. Percentage of predicted samples landing inside 0.5 units of truth
    margin = 0.5
    accuracy_within_margin = np.mean(np.abs(y_val - y_pred) <= margin) * 100

    print(f"RMSE (Root Mean Sq Error): {rmse:.4f}")
    print(f"MAE  (Mean Abs Error):     {mae:.4f}")
    print(f"R²   (Explained Variance): {r2:.4f}")
    print(f"Accuracy (Within ±{margin}):     {accuracy_within_margin:.2f}%\n")


🎯 Validation Performance
RMSE (Root Mean Sq Error): 0.5827
MAE  (Mean Abs Error):     0.4313
R²   (Explained Variance): 0.6789
Accuracy (Within ±0.5):     67.22%



## 6. Export Trained Model Artifact 

To avoid retraining when inferring, use `joblib.dump` to serialize the trained `XGBRegressor` instance object into the `models/` directory.

In [6]:
if df is not None:
    # Build complete destination
    model_out_path = WEIGHTS_DIR / "esa_soc_model.pkl"
    
    # Store binary using Joblib
    joblib.dump(model, model_out_path)
    
    print(f"📦 Trained model successfully saved to:")
    print(f"-> {model_out_path}")

📦 Trained model successfully saved to:
-> C:\Users\justi\Downloads\ML project data\Terraflux\models\weights\esa_soc_model.pkl
